# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Logic nằm trong `src/` (viết và test ở local); notebook chỉ **chạy thí nghiệm** trên Colab T4. Đọc `README.md` và `GUIDE.md` trước.

## 0. Môi trường (Colab T4)

In [ ]:
!git clone https://github.com/duclongt23/K4-Track-4-Day10-VisionLanguageModel.git

%cd /content/K4-Track-4-Day10-VisionLanguageModel

!pip install -q -r requirements-colab.txt
!pip uninstall -y -q torchao   # torchao cũ của Colab làm peft lỗi khi gắn LoRA lên ViT (E3); lab không dùng torchao
!python -m pytest -q
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [ ]:
import os, sys, gc, json, importlib
sys.path.insert(0, os.getcwd())          # để import src/ và scripts/

def reload_src():
    """Sau khi `!git pull` sửa src/*.py: nạp lại module (Python 3.13 của Colab không còn %autoreload).
    Xong chạy lại các ô `from src... import ...` để lấy hàm mới."""
    for m in ("model_utils", "data_utils", "training", "inference", "experiment", "rewards"):
        if f"src.{m}" in sys.modules:
            importlib.reload(sys.modules[f"src.{m}"])

DATA, WORK, EVID = "data", "work", "report/evidence"
DRIVE = "/content/drive/MyDrive/lab_day10"   # adapter + run.json được chép sang đây sau mỗi lần train
os.makedirs(DRIVE, exist_ok=True)
!python -m pytest my_tests -q


## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from src.data_utils import load_split, split_stats, session_overlap, load_image, gt_pixel_boxes, draw_boxes, make_gallery

splits = {s: load_split(s, DATA) for s in ("train", "val", "test")}
for s, e in splits.items():
    print(s, split_stats(e))
print("session nằm ở >= 2 split (phải rỗng):", session_overlap(splits))
print("dataset_version:", json.load(open(f"{DATA}/splits/meta.json"))["dataset_version"])

# 6 ảnh có box (xanh = nhãn đúng): kiểm tra bằng mắt là bạn hiểu nhãn
sample = [e for e in splits["train"] if len(e["objects"]) >= 2][::97][:6]
make_gallery([draw_boxes(load_image(e, DATA), gt_pixel_boxes(e)) for e in sample])

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from src.data_utils import objects_to_answer, answer_to_pixels, build_prompt

e = sample[0]
answer = objects_to_answer(e["objects"])
print(build_prompt("norm1000"), "\n")
print(answer)

# quy ngược JSON về pixel rồi vẽ lại (đỏ) lên box đúng (xanh): hai màu phải trùng nhau
back = answer_to_pixels(answer, "norm1000", e["width"], e["height"])
for g, b in zip(sorted(gt_pixel_boxes(e), key=lambda x: (x["bbox"][1], x["bbox"][0])), back):
    assert max(abs(a - c) for a, c in zip(g["bbox"], b["bbox"])) < 1.0, (g, b)
draw_boxes(load_image(e, DATA), gt_pixel_boxes(e), back)

## 3. E0: zero-shot

In [ ]:
from src.data_utils import load_split
from src.model_utils import load_model
from src.inference import run_inference, score_run
from scripts.score import parse_output

model, processor = load_model()
val40 = load_split("val", DATA, limit=40)                       # gỡ lỗi trên 40 ảnh val
p = run_inference(model, processor, val40, f"{WORK}/debug/e0_val40.jsonl", DATA)
raws = [json.loads(l)["raw"] for l in open(p, encoding="utf-8")]
print("parse_valid (40 ảnh val):", sum(parse_output(r) is not None for r in raws) / len(raws))
for r in raws[:5]:
    print("-" * 60, "\n", r)

In [ ]:
test = load_split("test", DATA)
p = run_inference(model, processor, test, f"{WORK}/e0.jsonl", DATA, resume=True)   # resume: Colab bị ngắt vẫn chạy tiếp được
r0 = score_run(p, "E0", f"{EVID}/e0.json")

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from src.training import encode_example, check_example, collate

e = sample[0]
enc = encode_example(processor, load_image(e, DATA), build_prompt("norm1000"), objects_to_answer(e["objects"]))
n_img, learned = check_example(processor, enc)
assert n_img == 220, n_img                                       # ảnh 640x360 = 220 token vision
assert learned.replace("<|im_end|>", "") == objects_to_answer(e["objects"])

del model; gc.collect()
import torch; torch.cuda.empty_cache()

## 5. Smoke test setup nặng nhất (E3) trước khi chạy thật
`debug=True`: 5 bước train + 8 ảnh val. Xem log `setup:` (252 / 348 module LoRA, số tham số huấn luyện), VRAM đỉnh và bằng chứng nạp lại projector. Kết quả ghi vào `work/debug`, không vào `report/evidence`.

In [ ]:
from src.training import load_config
from src.experiment import run_experiment

run_experiment(load_config("configs/e3.yaml"), "e3", copy_to=None, debug=True)

## 6. Vòng huấn luyện E1, E2, E3
Mỗi ô gọi `run_experiment`: train (lưu adapter + projector, chép sang Drive, ghi `run.json`) → nạp lại từ adapter → suy luận test → chấm kèm `--meta`. **Chỉ chấm `test` một lần cho mỗi setup.**

In [ ]:
cfg1 = load_config("configs/e1.yaml")
r1 = run_experiment(cfg1, "e1", copy_to=f"{DRIVE}/e1")

In [ ]:
cfg2 = load_config("configs/e2.yaml")
r2 = run_experiment(cfg2, "e2", copy_to=f"{DRIVE}/e2")

In [ ]:
cfg3 = load_config("configs/e3.yaml")
r3 = run_experiment(cfg3, "e3", copy_to=f"{DRIVE}/e3")

## 7. Suy luận và chấm E1–E3: bảng tổng hợp
Số liệu lấy từ `report/evidence/*.json` do `score.py` sinh ra, dùng để điền `REPORT.md`.

In [ ]:
import glob, pandas as pd

def summary():
    rows = []
    for f in sorted(glob.glob(f"{EVID}/e*.json")):
        d = json.load(open(f, encoding="utf-8"))
        m, pc = d["meta"], d["per_class"]
        rows.append({"file": os.path.basename(f), "params": m.get("trainable_params"), "vram_gb": m.get("peak_vram_gb"),
                     "phút": m.get("train_minutes"), "valid": round(d["parse_valid_rate"], 3),
                     "F1": round(d["micro"]["f1"], 3), "CI95": [round(x, 3) for x in d["f1_ci95"]],
                     "FP/ảnh": round(d["fp_per_image"], 2), "F1 pothole": round(pc["pothole"]["f1"], 3),
                     "F1 crack": round(pc["crack"]["f1"], 3), "F1 manhole": round(pc["manhole"]["f1"], 3)})
    return pd.DataFrame(rows)

summary()

## 8. E4: ablation (đổi ĐÚNG MỘT biến)
Chọn `BEST` là setup tốt nhất trong E1–E3. Mặc định ở dưới là `data_size` (10% so với 100%): chỉ cần một lần train mới, vì 100% chính là setup gốc.

In [ ]:
import copy
BEST_CFG, BEST_TAG = "configs/e3.yaml", "e3"        # sửa theo kết quả E1–E3 của bạn

cfg4 = copy.deepcopy(load_config(BEST_CFG))
cfg4["train_fraction"] = 0.1                         # biến duy nhất bị đổi
r4 = run_experiment(cfg4, "e4_data_size_0.1", copy_to=f"{DRIVE}/e4_data_size_0.1")

In [ ]:
# Giá trị còn lại (1.0) chính là setup gốc: dùng lại bằng chứng đã có, ghi rõ tên để báo cáo khớp
d = json.load(open(f"{EVID}/{BEST_TAG}.json", encoding="utf-8"))
d["name"] = "E4_DATA_SIZE_1.0"
json.dump(d, open(f"{EVID}/e4_data_size_1.0.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
summary()

### Kiểm chứng câu 3: chấm output thang 0–1000 như thể là pixel
Cùng một file dự đoán, chỉ đổi `--coord`. Kết quả ghi ra `work/debug`, **không** nằm trong `report/evidence`.

In [ ]:
r_pix = score_run(f"{WORK}/e1.jsonl", "E1_as_pixel", f"{WORK}/debug/e1_as_pixel.json", coord="pixel")
print("F1 đúng thang:", round(r1["micro"]["f1"], 3), "| F1 chấm sai thang:", round(r_pix["micro"]["f1"], 3))

## 9. Ảnh minh hoạ (≥ 6 ảnh đúng/sai) cho báo cáo câu 9

In [ ]:
from scripts.score import _match, _gt_boxes
from scripts.labcommon import read_jsonl
from src.data_utils import render_prediction

PRED = f"{WORK}/e3.jsonl"                           # file dự đoán của setup bạn muốn phân tích
preds = {r["image_id"]: r["raw"] for r in read_jsonl(PRED)}

def errors(e):                                      # FP + FN của một ảnh (chuẩn của score.py)
    per, _ = _match(_gt_boxes(e), parse_output(preds[e["image_id"]]) or [])
    return sum(v[1] + v[2] for v in per.values())

with_gt = [e for e in test if e["objects"]]
ranked = sorted(with_gt, key=errors)
picks = ranked[:3] + ranked[-3:]                    # 3 ảnh tốt nhất, 3 ảnh tệ nhất
gallery = make_gallery([render_prediction(e, preds[e["image_id"]], DATA) for e in picks], cols=3)
gallery.save(f"{EVID}/gallery_e3.png")
gallery

## 10. E5 (tuỳ chọn): reward IoU
Hàm `src/rewards.py` có test ở `my_tests/test_rewards.py`. Phần chạy GRPO với `trl` chưa được viết.

In [ ]:
!python -m pytest my_tests/test_rewards.py -q

## 11. Kiểm tra trước khi nộp

In [ ]:
!python scripts/check_submission.py